# Slowly Changing Dimensions (SCD)

## Objective

Implement and compare SCD Type 1 and SCD Type 2 strategies.

### Learning Goals

- SCD Type 1
- SCD Type 2
- History preservation
- Effective date management

### Source

- PostgreSQL customer changes
- bronze_customers

## SCD Type 1

Type 1 updates overwrite existing values.

Historical values are not preserved.

### Example

Hyderabad → Bengaluru

Only Bengaluru remains.

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    city,
    loyalty_points
FROM workspace.default.bronze_customers
WHERE customer_id = 1001
-- What you're seeing right now is Type 1 behavior.Old City = Lost,New City = Stored.

## Create SCD Type 2 Structure

Type 2 preserves historical changes by creating new records instead of overwriting existing records.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.dim_customers_scd2
(
    customer_id BIGINT,
    customer_name STRING,
    city STRING,
    membership_type STRING,
    customer_status STRING,
    loyalty_points BIGINT,

    effective_from TIMESTAMP, -- When record became active
    effective_to TIMESTAMP,  -- When record became inactive

    is_current BOOLEAN -- Flag to indicate if record is current
);

## Load Initial Customer Version

In [0]:
%sql
INSERT INTO workspace.default.dim_customers_scd2
SELECT
    customer_id,
    customer_name,
    city,
    membership_type,
    customer_status,
    loyalty_points,

    current_timestamp() AS effective_from,

    CAST(NULL AS TIMESTAMP) AS effective_to,

    TRUE AS is_current

FROM workspace.default.bronze_customers;

In [0]:
%sql
SELECT *
FROM workspace.default.dim_customers_scd2

## Simulate Customer Change

A customer record has been updated in the operational PostgreSQL system.

The updated record will be used to demonstrate SCD Type 2 processing.

In [0]:
jdbc_url = (
    "jdbc:postgresql://"
    "ep-hidden-scene-bind3jz7-pooler.c-5.ap-southeast-1.aws.neon.tech:5432/neondb"
)

In [0]:
scd_incremental_df = (
    spark.read
         .format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", "customers")
         .option("user", "neondb_owner")
         .option("password", "npg_UnJmC6vfAi8K")
         .option("driver", "org.postgresql.Driver")
         .load()
         .filter("customer_id = 1001")
)

In [0]:
scd_incremental_df.createOrReplaceTempView(
    "customer_updates"
)

In [0]:
%sql
SELECT *
FROM customer_updates

## Expire Existing Version

Mark the current customer record as historical.


In [0]:
%sql
UPDATE workspace.default.dim_customers_scd2
SET
    effective_to = current_timestamp(),
    is_current = FALSE
WHERE customer_id = 1001
AND is_current = TRUE

## Insert New Version

Create a new current customer record representing the latest state.

In [0]:
%sql
INSERT INTO workspace.default.dim_customers_scd2
SELECT
    customer_id,
    customer_name,
    city,
    membership_type,
    customer_status,
    loyalty_points,

    current_timestamp() AS effective_from,

    CAST(NULL AS TIMESTAMP) AS effective_to,

    TRUE AS is_current

FROM customer_updates

In [0]:
%sql
SELECT
    customer_id,
    city,
    loyalty_points,
    effective_from,
    effective_to,
    is_current
FROM workspace.default.dim_customers_scd2
WHERE customer_id = 1001
ORDER BY effective_from

## Summary

### Concepts Covered

✅ SCD Type 1

✅ SCD Type 2

✅ Effective Dates

✅ Current Flag

✅ Historical Tracking

### Key Difference

Type 1 overwrites history.

Type 2 preserves history by creating new versions.

### Output

- workspace.default.dim_customers_scd2

✅ SCD Processing Completed
